In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from pprint import pprint

from langchain.messages import AIMessage, HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user is inquiring about various aspects of a fictional place called Lunapolis, including its capital, weather, population of cheese miners, and the potential for a union strike.\n\n## SUMMARY\n\n- The capital of the moon is identified as Lunapolis.\n- The weather in Lunapolis is reported as clear skies with a high temperature of 120°C and a low of -100°C.\n- Lunapolis has a population of 100,000 cheese miners.\n- The cheese miners' union is expected to strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='13ed7edb-2537-4552-b42d-a81ea210891c'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='71f90bac-e544-49a5-8c89-c82f5cf1266b'

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT

The user is inquiring about various aspects of a fictional place called Lunapolis, including its capital, weather, population of cheese miners, and the potential for a union strike.

## SUMMARY

- The capital of the moon is identified as Lunapolis.
- The weather in Lunapolis is reported as clear skies with a high temperature of 120°C and a low of -100°C.
- Lunapolis has a population of 100,000 cheese miners.
- The cheese miners' union is expected to strike due to dissatisfaction with the new president.

## ARTIFACTS

None

## NEXT STEPS

None


In [6]:
print(response["messages"][-1].content)

Since Lunapolis is a fictional setting, here’s a constructive, peace-oriented way I’d respond as the new president to the cheese miners’ union. The emphasis is on dialogue, safety, fairness, and keeping essential operations running.

High-level approach
- Acknowledge and validate concerns: safety, fair pay, predictable hours, and a voice in decisions that affect workers.
- Commit to open, good-faith negotiations with a structured process.
- Prioritize safety in extreme conditions (high heat) and invest in humane, sustainable working conditions.
- Seek a win-win outcome that preserves cheese production, supports workers, and strengthens Lunapolis in the long run.

Immediate steps (within the first 72 hours)
- Publicly invite the union to negotiate in good faith and set a concrete timeline (e.g., first negotiation session within 5–7 days).
- Establish a joint negotiation committee with equal representation from the administration and the union, plus a neutral mediator if requested.
- Pub

## Trim/delete messages

In [ ]:
from typing import Any

from langchain.agents import AgentState
from langchain.agents.middleware import before_agent
from langchain.messages import RemoveMessage, ToolMessage
from langgraph.runtime import Runtime


@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state before the agent is invoked."""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],    # Add the trim_messages middleware to the agent
)

In [9]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='de48b7f9-28a7-441f-947a-8a6b2b820328'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='cbdedd7d-78f0-4b4c-9727-5098a41aa820', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='387f1c90-792a-43af-889d-4ff99fffe9d2'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='ae8603a8-a408-47d2-ba8c-9ef83115a90a', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='320ba7dc-486f-465a-8231-14f8da26c3a7'),
              AIMessage(content='I can’t read the device’s temperature from here. If you’re wor

In [10]:
print(response["messages"][-1].content)

I can’t read the device’s temperature from here. If you’re worried about overheating, here’s how to check and next steps:

What to do now
- Feel the exterior gently. If it’s hot to the touch (especially around vents or the battery area), power it off and unplug it. Let it cool in a well-ventilated area before trying again.
- Let it cool for 15–30 minutes. Do not try to force it on if it’s hot.
- Check for blocked vents and dust. If the fans or vents are clogged, it can overheat and fail to boot.
- Try a different power source or charger (same wattage/spec). If the device has a removable battery, remove it, then plug in power and try to boot with AC only.
- Do a basic power reset: hold the power button for 10–15 seconds to force a shutdown, then press once to try to turn it on again.

If you still can’t turn it on
- Look for any lights or beeps when you try to power it on and tell me what you see (color, pattern, duration). This can help diagnose a power or motherboard issue.
- If the d